In [0]:
dbutils.widgets.text("fPath_Src", "/Volumes/yourtcat/schemayourt/volyourt/animal_data_dirty1", "Input File Path");
dbutils.widgets.text("fPath_Dest", "/Volumes/yourtcat/schemayourt/volyourt/G", "Gold Layer");

fPath_Src =dbutils.widgets.get("fPath_Src");

df = spark.read.csv(
    fPath_Src,
    header=True,
    sep=";",  # ✓ Use 'sep' not 'delimiter'
    inferSchema=True
)

In [0]:
from pyspark.sql.functions import col
display(df);
#display(df.filter(col('Country').contains('Hungary')));


In [0]:
import requests
from pyspark.sql.functions import col, count, when, isnan, regexp_replace, trim, round, monotonically_increasing_id, upper, initcap, split,lit ,udf, expr, to_timestamp, date_format, lower, concat, coalesce, try_to_timestamp, try_to_date
from pyspark.sql.types import StringType, DoubleType, FloatType, IntegerType, DecimalType, DatetimeType, DateType, TimestampType

gPath =dbutils.widgets.get("fPath_Dest");


# Define UDF to fetch currency from API
@udf(returnType=StringType())
def get_currency_code(country_name):
    try:
        # Using restcountries.com free API
        response = requests.get(f"https://restcountries.com/v3.1/name/{country_name}")
        if response.status_code == 200:
            data = response.json()
            if data and len(data) > 0:
                currencies = data[0].get('currencies', {})
                if currencies:
                    return list(currencies.keys())[0]  # Get first currency code
        return "Unknown"
    except:
        return "Unknown"

# ========== STEP 1: Count null values per column ==========
null_counts = []
for column_name in df.columns:    
    null_counts.append((column_name, df.filter(col(column_name).isNull()).count() ))
# Create dataframe with null counts
null_df = spark.createDataFrame(null_counts, ["Column_Name", "Null_Count"])
print("=== NULL VALUE COUNTS ===")
display(null_df);

# ========== STEP 2: Remove rows with nulls in critical columns ==========
#following code works, tested and gives results as the dfcleanedsql
#df_cleaned = df.filter(
#    col('Animal type').isNotNull() & 
#    (col('Animal type') != "") &
#    col('Latitude').isNotNull() & 
#    col('Longitude').isNotNull() & 
#    col('Animal name').isNotNull() &
#    (col('Animal name') != "") );
df_cleanedsql = df.filter(""" (`Animal type` is not null and `Animal type` !="") and Latitude is not null and Longitude is not null and (`Animal name` is not null and `Animal name` !='')
                        """ );
print("Before Cleaning");
display(df_cleanedsql);


# ========== STEP 3: Clean string columns ==========
# Keep only: alphabets (A-Z, a-z), spaces, and French characters (àâäéèêëïîôùûüÿçœæ)
for column_name in df_cleanedsql.columns:
    col_type = df_cleanedsql.schema[column_name].dataType
    
    if (isinstance(col_type, StringType) and column_name != "Observation date") :
        # Remove special characters, keep alphabets, spaces, and French chars
        df_cleanedsql = df_cleanedsql.withColumn(
            column_name,
            trim(regexp_replace(col(column_name), "[^a-zA-Z0-9 àâäéèêëïîôùûüÿçœæÀÂÄÉÈÊËÏÎÔÙÛÜŸÇŒÆ]", ""))
        )
    elif (isinstance(col_type, StringType) and column_name == "Observation date") :
        df_cleanedsql = df_cleanedsql.withColumn(
            column_name,
            trim(regexp_replace(col(column_name), "[^a-zA-Z0-9 àâäéèêëïîôùûüÿçœæÀÂÄÉÈÊËÏÎÔÙÛÜŸÇŒÆ.-/]", ""))
        )

# ========== skipping STEP 4, not needed: Round numeric columns to 4 decimals  ==========
#
#for column_name in df_cleaned.columns:
#    col_type = df_cleaned.schema[column_name].dataType
#    
#    if isinstance(col_type, (DoubleType, FloatType, DecimalType)):
#        df_cleaned = df_cleaned.withColumn(
#            column_name,
#            round(col(column_name), 4)
#        )

# ========== STEP 5: Add ID for 'Animal type' column ==========
# Create unique IDs for each animal type
df_cleanedsql = df_cleanedsql.withColumn("Animal_Type_ID", monotonically_increasing_id()+1001);df_cleanedsql = df_cleanedsql.withColumn('Data compiled by', upper(trim(col('Data compiled by'))));

df_Author = df_cleanedsql.select('Data compiled by','Gender' ).distinct() \
.withColumnRenamed('Data compiled by', 'Author') \
.withColumn('Author_ID', monotonically_increasing_id()+10001) \
.withColumn('AuthorWithTitle',
         when(lower(col('Gender')) == 'male', concat(lit('Mr. '), col('Author'))) \
        .when(lower(col('Gender')) == 'female', concat(lit('Ms. '), col('Author'))) \
        .otherwise(col('Author')) \
        ) \
.withColumn('Author FirstName', split(col('Author'), ' ').getItem(0)) \
    .withColumn('Author LastName', split(col('Author'), ' ').getItem(1)) \
.drop('Gender');
df_cleanedsql = df_cleanedsql.join(df_Author, col('Data compiled by') == df_Author['Author'], how='left');
df_cleanedsql = df_cleanedsql.drop('Data compiled by').drop('Author').withColumnRenamed('Author_ID', 'CAuthor_ID');


# ========== STEP 3: Clean specific columns ==========

df_cleanedsql = df_cleanedsql \
    .withColumn('Animal type', initcap(col('Animal type'))) \
    .withColumn('Country', 
        when(col('Country') == 'DE', 'Germany')
        .when(col('Country') == 'FR', 'France')
        .when(col('Country') == 'IT', 'Italy')
        .when(col('Country') == 'ES', 'Spain')
        .when(col('Country') == 'UK', 'United Kingdom')
        .when(col('Country') == 'NL', 'Netherlands')
        .when(col('Country') == 'BE', 'Belgium')
        .when(col('Country') == 'AT', 'Austria')
        .when(col('Country') == 'CH', 'Switzerland')
        .when(col('Country') == 'DK', 'Denmark')
        .when(col('Country') == 'SE', 'Sweden')
        .when(col('Country') == 'NO', 'Norway')
        .when(col('Country') == 'FI', 'Finland')
        .when(col('Country') == 'PL', 'Poland')
        .when(col('Country') == 'CZ', 'Czech Republic')
        .when(col('Country') == 'HU', 'Hungary')
        .when(col('Country') == 'PT', 'Portugal')
        .when(col('Country') == 'GR', 'Greece')
        .when(col('Country') == 'IE', 'Ireland')
        .otherwise(col('Country'))
    ) \
    .withColumn('Gender', initcap(col('Gender'))) \
    .withColumn('Currency', get_currency_code(col('Country'))) \
    .withColumn( 
    'body length inches',
    round(col('body length cm') * 0.393701, 2)
    ) \
    .withColumn("Observation D",coalesce(
        try_to_date(col('Observation date'), lit("dd.MM.yyyy")),
        try_to_date(col('Observation date'), lit("dd.MM.yyyy")),
        try_to_date(col('Observation date'), lit("dd MMM yyyy")),
        try_to_date(col('Observation date'), lit("d MMM yyyy")),
        try_to_date(col('Observation date'), lit("yyyy-MM-dd")),
        try_to_date(col('Observation date'), lit("MM/dd/yyyy"))
    )) \
    .withColumn("Observation DT",     
        col('Observation D').cast("timestamp")
    ) \

display(df_Author);

df_cleanedsql.write.mode("overwrite").option("header", True).option("sep", ",").csv(
    gPath+ "/CSV" 
);
df_cleanedsql.write.parquet(gPath+ "/Parquet", mode="append");

df_Author.write.mode("append").option("header", True).option("sep", ",").csv(
    gPath+ "/CSV"
);

#Replace chars that Avro don't like, i think same apply to delta
df_cleanedsql = df_cleanedsql.toDF(*[
    c.strip().lower().replace(" ", "_")
    for c in df_cleanedsql.columns
]);
display(df_cleanedsql);
df_cleanedsql.write.mode("append").format("delta").save(gPath+ "/Delta");
df_cleanedsql.write.mode("append").format("avro").save(gPath+ "/Avro");    


In [0]:
from pyspark.sql.functions import col, count, when, isnan, regexp_replace, trim, round, monotonically_increasing_id, upper, initcap, split,lit ,udf, expr, to_timestamp, date_format, lower, concat, coalesce, try_to_timestamp, try_to_date, current_timestamp
from pyspark.sql.types import StringType, DoubleType, FloatType, IntegerType, DecimalType, DatetimeType, DateType, TimestampType
from delta.tables import DeltaTable

gPath =dbutils.widgets.get("fPath_Dest");
df_goldCSV = spark.read.csv(
    gPath+ "/CSV",
    header=True,
    sep=",",  # ✓ Use 'sep' not 'delimiter'
    inferSchema=True
);
df_avro = spark.read.format("avro").load(gPath+ "/Avro");
#DELTA TBLE TIME TRAVELING ADVANTAGE WITH SCD AS PER GENIE
delpath = gPath+ "/Delta"
his_df_delta = spark.sql(f"DESCRIBE HISTORY delta.`{delpath}`")
print(f"Displaying history");
display(his_df_delta);
#reading previous version as per 
df_del_0 = spark.read.format("delta").option("versionAsOf", 0).load(delpath);
print(f"Displaying Version 0 of deltatable");
display(df_del_0);
#bytime 
print(f"Displaying Timestampasof 2026-05-08 23:37:25 of deltatable, another way");
df_del_yesterday = spark.read.format("delta").option("timestampAsOf", "2026-05-08 23:37:25").load(delpath);
display(df_del_yesterday);
#comparing versions 
df_del_Curr = spark.read.format("delta").load(delpath);
print(f"Current version rows:{df_del_Curr.count()}");
print(f"prev version rows:{df_del_yesterday.count()}");
display(df_del_Curr);

    # ========== RESTORE TO PREVIOUS VERSION, Not tested as i don't have versions yet ==========
    # Rollback to version 0 if something went wrong
    #spark.sql(f"RESTORE TABLE delta.`{delta_path}` TO VERSION AS OF 0")

    # ========== VACUUM OLD FILES (cleanup), Not tested as don't have older versions==========
    # Remove files older than 7 days (default retention)
    #spark.sql(f"VACUUM delta.`{delta_path}` RETAIN 168 HOURS")

#SCD TYPE1 (MEANS OLD OVERWRITTEN WITH NEW)
    #simulating new data with updates
new_data = [("Dog", "Berlin", "Germany", 52.51, 13.42,45.5, "2026-05-02", "2026-05-02 00:00:00"),("Cat", "Paris", "France", 48.8566, 2.3522, 25.3, "2026-05-02" , "2026-05-02 00:00:00"),("Elephant", "Nairobi", "Kenya", -1.2921, 36.8219, 320.0, "2026-05-02", "2026-05-02 00:00:00") 
]
new_df = spark.createDataFrame(new_data, 
    ["animal_type", "city", "country", "latitude", "longitude", "body_length_cm", "observation_d", "observation_dt"]
)
del_tbl = DeltaTable.forPath(spark, delpath)
del_tbl.alias("t").merge( new_df.alias("s"), "t.animal_type = s.animal_type").whenMatchedUpdate(
    set={
        "country": "s.country",
        "latitude": "s.latitude",
        "longitude": "s.longitude",
        "body_length_cm": "s.body_length_cm",
        "observation_date": "s.observation_d",
        "observation_dt": "s.observation_dt"
    }
).whenNotMatchedInsert(
    values={
        "animal_type": "s.animal_type",
        "country": "s.country",
        "latitude": "s.latitude",
        "longitude": "s.longitude",
        "body_length_cm": "s.body_length_cm",
        "observation_d": "s.observation_d",
        "observation_dt": "s.observation_dt"
    }
).execute();

#view result
display(spark.read.format("delta").load(delpath));

#SCD Type 2
del_scd2 = (del_tbl.toDF()).withColumn("modified_date", current_timestamp()).withColumn("is_current", lit(True) ).withColumn("version",lit(1))
#initial load
del_scd2.write.mode("overwrite").save(gPath+ "/DeltaSCD2");
#now simulatin updated data to show scd2
upd_data = del_scd2;
del_scd2 = DeltaTable.forPath(spark, gPath+ "/DeltaSCD2");
#upd_data = [("Dog",  "Germany", 52.1235,13.134, 45.5, "2026-05-02","2026-05-02 00:00:00"),("Cat", "France", 48.8566, 2.3522, 25.3, "2026-05-02","2026-05-02 00:00:00"),("Elephant", "Kenya", 11.2921, 36.8219, 90.0, "2026-05-02", "2026-05-02 00:00:00")];
#upd_df = spark.createDataFrame(upd_data, ["animal_type", "country", "latitude", "longitude", "body_length_cm", "observation_d", "observation_dt"]) \
#.withColumn("modified_date", current_timestamp()) \
#.withColumn("is_current", lit(True) ) \
#.withColumn("version",lit(2))

upd_df = upd_data \
    .filter((col("animal_type").isin("Dog", "Cat", "Elephant")) & (col("is_current") == True)) \
    .withColumn("body_length_cm", lit(90)) \
    .withColumn("country", lit("Germany")) \
    .withColumn("modified_date", current_timestamp()) \
    .withColumn("version", lit(2)) \
    .withColumn("is_current", lit(True))
del_scd2.toDF().printSchema();
upd_df.printSchema();

del_scd2.alias("t").merge( upd_df.alias("d"), "t.animal_type = d.animal_type and t.country = d.country and t.is_current = true").whenMatchedUpdate(
    set = {
        "modified_date": "d.modified_date",
        "is_current": "false"
    }
).execute();

upd_df.write.format("delta").mode("append").option("mergeSchema", "true").save(gPath +"/DeltaSCD2");
#view history now
result_scd2 = spark.read.format("delta").load(gPath+ "/DeltaSCD2");
display(result_scd2.orderBy("animal_type", "version"));
display(result_scd2.filter(col("is_current")== True));
#historical
display(result_scd2.filter(col("is_current")== False) );
print(f"Displaying history");
dscd2path = gPath+ "/DeltaSCD2";
display(spark.sql(f"DESCRIBE HISTORY delta.`{dscd2path}`"));

print("\n=== STORAGE SIZE COMPARISON ===")
dbutils.fs.ls(gPath + "/CSV")
dbutils.fs.ls(gPath + "/Parquet")
dbutils.fs.ls(gPath + "/Avro")
dbutils.fs.ls(gPath + "/Delta")

#df_pq = spark.read.format("parquet").load(gPath+ "/Parquet");

#display(df_goldCSV);
#display(df_avro);
#display(df_delta);
#display(df_pq);